# End-to-End Test: V5 Environment with DoubleDQN and Bybit

Этот ноутбук проверяет интеграцию новой среды `TradingEnvV5` с агентом `DoubleDQNAgent`, используя стандартные функции обучения и бенчмарка.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import clear_output

## 1. Загрузка данных и генерация фичей

In [ ]:
from core.data.data_loader import load_crypto_data
from core.features.feature_generator import FeatureGenerator

df = load_crypto_data(symbol='BTCUSDT', start_date='2024-01-01', interval='4h', source='bybit_futures')

feature_generator = FeatureGenerator()
data_features = feature_generator.transform(df)
print(f"Features shape: {data_features.shape}")

## 2. Инициализация TradingEnvV5

In [ ]:
from custom_envs.trading_env_v5 import TradingEnvV5

# Отключаем continuous_action для работы с DQN
train_env = TradingEnvV5(
    df=data_features,
    continuous_action=False,
    t_max=1000
)

print(f"Action space: {train_env.action_space}")
print(f"Observation space: {train_env.observation_space.shape}")

## 3. Обучение DoubleDQNAgent через train_dqn_agent

In [ ]:
from agents.double_dqn_agent import DoubleDQNAgent
from utils.train_dqn_agent import train_dqn_agent

agent = DoubleDQNAgent.from_env(train_env, lr=5e-4, batch_size=64)

epsilon_start = 1.0
epsilon_end = 0.1

rewards, pnl_per_episode, eval_metrics = train_dqn_agent(
    train_env, 
    agent, 
    num_episodes=50,  # Для теста
    target_update_freq=None,
    adaptive_epsilon=True, 
    epsilon_max=epsilon_start, 
    epsilon_min=epsilon_end,
    eval_every=10,
    verbose=True
)

## 4. Benchmark: Обученный DQN vs Случайный Агент

In [ ]:
from agents.random_agent import RandomAgent
from utils.train_dqn_agent import evaluate_agent

# Создаем тестовую среду (например, берем другой кусок данных или просто без t_max)
test_env = TradingEnvV5(
    df=data_features.iloc[-2000:].reset_index(drop=True),
    continuous_action=False,
    t_max=1500
)

random_agent = RandomAgent(test_env.action_space)

print("Оцениваем Random Agent...")
rand_reward, rand_pnl = evaluate_agent(random_agent, test_env, num_episodes=5)
print(f"Random Agent -> Mean Reward: {rand_reward:.4f}, Mean PnL: {rand_pnl:.2f}%")

print("\nОцениваем DoubleDQN Agent...")
dqn_reward, dqn_pnl = evaluate_agent(agent, test_env, num_episodes=5)
print(f"DoubleDQN Agent -> Mean Reward: {dqn_reward:.4f}, Mean PnL: {dqn_pnl:.2f}%")

## 5. Визуализация действий на графике (Свечи)

In [ ]:
from core.visualization.visualizer_candles import CandleChartVisualizer

candle_visualizer = CandleChartVisualizer(use_volume_width=False)

# Собираем действия DQN на тестовой среде
obs, _ = test_env.reset()
done = False
dqn_actions = []

while not done:
    possible_actions = test_env.get_possible_actions()
    action = agent.act(obs, possible_actions=possible_actions, training=False)
    obs, _, done, _, _ = test_env.step(action)
    
    # action в V5 дискретной версии это список [тип(0-hold, 1-buy, 2-sell), размер_позиции]
    dqn_actions.append(int(action[0]))

TOP_N = 100
print("DQN Agent Actions on Test Data:")
candle_visualizer.plot_candlestick(
    test_env.df.iloc[test_env.start_index : test_env.start_index + TOP_N].reset_index(drop=True),
    actions=dqn_actions[:TOP_N]
)